# Project Capstone

## Import Libraries

In [ ]:
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

## Data Wrangling

In [ ]:
data = pd.read_csv("./data/data.csv")
data = pd.DataFrame(data)
data.head()

In [ ]:
data.info()

In [ ]:
data.isna().sum()

In [ ]:
data.fillna(value="None", inplace=True)

In [ ]:
data.info()

In [ ]:
print("Jumlah duplikasi: ", data.duplicated().sum())

In [ ]:
dataset = data.drop(columns=['Person ID', 'Occupation', 'Blood Pressure'])
dataset.head()

In [ ]:
dataset.describe()

In [ ]:
label_encoders = {}
for column in dataset.select_dtypes(include=['object']).columns:
    label_encoders[column] = LabelEncoder()
    dataset[column] = label_encoders[column].fit_transform(dataset[column])

In [ ]:
dataset.describe()

In [ ]:
x = dataset.drop(columns=['Quality of Sleep'])
y = dataset['Quality of Sleep']

In [ ]:
print(x.iloc[0])

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.3, random_state=42)

print("Training data size:", len(x_train))
print("Validation data size:", len(x_val))

print("\nTraining label size:", len(y_train))
print("Validation label size:", len(y_val))

In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)

In [ ]:
print(x_train_scaled[0])
print(x_val_scaled[0])

## Machine Learning

In [ ]:
class CustomCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs['val_mae'] <= 0.1:
            print("Early stopping triggered at epoch ", epoch, " because validation MAE is less than 0.012")
            self.model.stop_training = True

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='linear')
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_squared_error',
    metrics=['mae']
)

In [ ]:
history = model.fit(x_train_scaled, y_train,
                    epochs=500,
                    validation_data=(x_val_scaled, y_val),
                    batch_size=32,
                    callbacks=[CustomCallback()]
)

In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']
mae = history.history['mae']
val_mae = history.history['val_mae']

# Create a range for epochs
epochs = range(1, len(loss) + 1)

# Plot the loss over epochs
plt.figure(figsize=(14, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(epochs, loss, label='Training Loss')
plt.plot(epochs, val_loss, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()

# MAE plot
plt.subplot(1, 2, 2)
plt.plot(epochs, mae, label='Training MAE')
plt.plot(epochs, val_mae, label='Validation MAE')
plt.title('Training and Validation MAE')
plt.xlabel('Epochs')
plt.ylabel('Mean Absolute Error')
plt.legend()

plt.show()

In [ ]:
print(x.iloc[0])
print(x_train_scaled[0])
print(y.iloc[0])

In [ ]:
predictions = model.predict(x_val_scaled)

print("Beberapa prediksi pertama:")
for i in range(10):
    print(f"Prediksi: {predictions[i][0]:.2f}, Nilai aktual: {y_val.iloc[i]:.2f}")


In [ ]:
# model.save('model.h5')

## Loaded Model

In [ ]:
loaded_model = tf.keras.models.load_model('primary_model.h5')
loaded_model.summary()

In [ ]:
predictions = loaded_model.predict(x_train_scaled)

print("Beberapa prediksi pertama:")
for i in range(10):
    print(f"Prediksi: {predictions[i][0]:.2f}, Nilai aktual: {y_train.iloc[i]:.2f}")

## User Input

In [ ]:


data_input = {
    "Gender": "Male",
    "Age": 27,
    "Sleep Duration": 6.1,
    "Physical Activity Level": 42,
    "Stress Level": 6,
    "BMI Category": "Overweight",
    "Heart Rate": 77,
    "Daily Steps": 4200,
    "Sleep Disorder": "None"
}

data_user = pd.DataFrame([data_input])
dataset = pd.concat([dataset, data_user], ignore_index=True)

In [ ]:
# Data Encoding
for column in data_user.select_dtypes(include=['object']).columns:
    label_encoders[column] = LabelEncoder()
    data_user[column] = label_encoders[column].fit_transform(data_user[column])
    
x_input_scaled = scaler.transform(x_val)

print(x_input_scaled[-1])

In [ ]:
dataset.tail()

In [ ]:
predictions = loaded_model.predict(x_input_scaled[-1].reshape(1, -1))
print(f"Prediksi: {predictions[0][0]:.2f}")